In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
from tqdm import tqdm
import json
import warnings

warnings.filterwarnings('ignore')

## incs_no 기준 -> sess_id기준으로 데이터 변경 해야함!

In [19]:
# 추후 로그 나눠서 저장해놓고 쓰는걸로 바꾸기!
log_path = './data/LOG1.csv'
msg_path = './data/msg_indicate.csv'
cust_path = './data/CUST.csv'
output_path = './data/grp_sess.csv'

In [20]:
log = pd.read_csv(log_path)
print(log.shape)
log.head(2)

(6733659, 98)


,Unnamed: 0,STND_YMD,SITE_ID,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,GOOGLE_CID,SESS_ID,ADID,INCS_NO,...,DVCE_MKT_NM,DVCE_OPRT_NM,DVCE_OPRT_VER,NEW_VISIT_YN,STORE_CD,INVN_ST_YN,SESS_SN_STAY_TIME,EXPS_RSLT_LIST,LIST_TP_NM,ETL_PROC_DTTM
0,0,2024-06-27,UA-110770460-3,APP,d780c094f3a5c7f2959af19e83c535296d348c04b202e4...,2024-06-27 15:25:52.186,6651cd807786ada0dbe87474fcbe619d,6651cd807786ada0dbe87474fcbe619d1719469418,e00992c4-2146-4120-9357-021d9bf604e0,c8ef20aacdabd2aa358a8586f4c2bd4210509e33e564d7...,...,Galaxy Note10 5G,Android,Android 12,N,NaN,NaN,1.22,NaN,NaN,2024-11-27 18:26:32.797
1,1,2024-06-27,UA-110770460-3,APP,2abe35fc5e26480e5ec0ef900e2806f6aef772066745d4...,2024-06-27 10:49:40.005,7fca1b339adc77c666f040163ceb3b22,7fca1b339adc77c666f040163ceb3b221719452972,1b6050cd-d32f-4f38-8617-6df79efde63f,edecad485c2d71cc3b324e3547924a4dee66be8060f3f9...,...,Galaxy Note20 5G,Android,Android 13,N,NaN,NaN,0.60,NaN,NaN,2024-11-27 18:26:32.797


In [21]:
use_col_list = ['STND_YMD', 'WEB_APP_CL_CD', 'LOG_SEQ', 'LOG_DTTM', 'SESS_ID', 'INCS_NO', 'AGE', 'BRTH_YEAR', 
                'SEX_CD', 'EMP_YN', 'CUST_GRD_NM', 'DVCE_TP_CD', 'SITE_URL', 'PG_URL', 'PG_NM',
                'PG_TP_VL', 'SVC_CL_CD', 'UTM_SOURCE', 'ACCM_STAY_TIME', 'PG_STAY_TIME', 'EVNT_NM', 'EVNT_CAT_DTL',
                'PRD_INFO']

log = log[use_col_list]
print(log.shape)
log.head(2)

(6733659, 23)


,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,PG_URL,PG_NM,PG_TP_VL,SVC_CL_CD,UTM_SOURCE,ACCM_STAY_TIME,PG_STAY_TIME,EVNT_NM,EVNT_CAT_DTL,PRD_INFO
0,2024-06-27,APP,d780c094f3a5c7f2959af19e83c535296d348c04b202e4...,2024-06-27 15:25:52.186,6651cd807786ada0dbe87474fcbe619d1719469418,c8ef20aacdabd2aa358a8586f4c2bd4210509e33e564d7...,42,1982,F,N,...,https://m.innisfree.com/kr/ko/mMypageCouponZon...,쿠폰존,OTHERS,OTHERS,NaN,134,2,screen_view,NaN,NaN
1,2024-06-27,APP,2abe35fc5e26480e5ec0ef900e2806f6aef772066745d4...,2024-06-27 10:49:40.005,7fca1b339adc77c666f040163ceb3b221719452972,edecad485c2d71cc3b324e3547924a4dee66be8060f3f9...,29,1995,F,N,...,https://m.innisfree.com/kr/ko/mMypage.do,마이페이지 - 이니스프리 마이페이지 모바일 | INNISFREE,MY,MY,NaN,8,1,screen_view,NaN,NaN


In [22]:
log['EVNT_NM'].value_counts()

EVNT_NM
screen_view              3832968
view_promotion            716204
view_item                 688159
view_promotion_detail     605829
page_view                 521694
view_search_results       129260
view_item_list            116674
select_promotion           70628
purchase                   52243
Name: count, dtype: int64

### PRD_INFO 파싱

In [23]:
# JSON 형식의 문자열 딕셔너리로 변환하는 함수 정의
def json_str_to_dict(json_str):
    if isinstance(json_str, float) and np.isnan(json_str):
        return {} # 빈 딕셔너리 반환
    pattern = re.compile(r'\"(.*?)\":\s*\"(.*?)\",*\n*')
    matches = pattern.findall(json_str)
    result_dict = {key: (value if value != '(not set)' else None) for key, value in matches}
    return result_dict

# PRD_INFO 컬럼에 함수를 적용하여 파싱된 데이터를 새로운 칼럼으로 추가하는 함수 정의
def apply_and_expand(df, col_name):
    # 각 행의 데이터를 딕셔너리로 변환
    df_dicts = df[col_name].apply(json_str_to_dict)
    
    # 딕셔너리 형태의 데이터를 데이터프레임으로 변환
    expanded_df = pd.json_normalize(df_dicts)
    
    # 생성된 새로운 데이터프레임을 기존 데이터프레임에 병합
    result_df = pd.concat([df, expanded_df], axis=1)
    
    return result_df


# apply_and_expand 함수를 사용하여 데이터프레임을 업데이트
log_prse = apply_and_expand(log, 'PRD_INFO')
print(log_prse.shape)
log_prse.head(2)

(6733659, 44)


,STND_YMD,WEB_APP_CL_CD,LOG_SEQ,LOG_DTTM,SESS_ID,INCS_NO,AGE,BRTH_YEAR,SEX_CD,EMP_YN,...,prd_optn,prd_prom_id,prd_prom_nm,prd_qty,prd_tp_cat_vl,brnd_cd,prd_norm_prc,prd_sal_prc,acml_bt_pt,prd_sal_amt
0,2024-06-27,APP,d780c094f3a5c7f2959af19e83c535296d348c04b202e4...,2024-06-27 15:25:52.186,6651cd807786ada0dbe87474fcbe619d1719469418,c8ef20aacdabd2aa358a8586f4c2bd4210509e33e564d7...,42,1982,F,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-06-27,APP,2abe35fc5e26480e5ec0ef900e2806f6aef772066745d4...,2024-06-27 10:49:40.005,7fca1b339adc77c666f040163ceb3b221719452972,edecad485c2d71cc3b324e3547924a4dee66be8060f3f9...,29,1995,F,N,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
log_prse.columns

Index(['STND_YMD', 'WEB_APP_CL_CD', 'LOG_SEQ', 'LOG_DTTM', 'SESS_ID',
       'INCS_NO', 'AGE', 'BRTH_YEAR', 'SEX_CD', 'EMP_YN', 'CUST_GRD_NM',
       'DVCE_TP_CD', 'SITE_URL', 'PG_URL', 'PG_NM', 'PG_TP_VL', 'SVC_CL_CD',
       'UTM_SOURCE', 'ACCM_STAY_TIME', 'PG_STAY_TIME', 'EVNT_NM',
       'EVNT_CAT_DTL', 'PRD_INFO', 'ad_nm', 'ad_slot_nm', 'brnd_nm',
       'chn_prd_cd', 'chn_prd_nm', 'prd_category1', 'prd_category2',
       'prd_category3', 'prd_category4', 'prd_cpn_nm', 'prd_loc', 'prd_optn',
       'prd_prom_id', 'prd_prom_nm', 'prd_qty', 'prd_tp_cat_vl', 'brnd_cd',
       'prd_norm_prc', 'prd_sal_prc', 'acml_bt_pt', 'prd_sal_amt'],
      dtype='object')

### 로그데이터 그래프 임베딩
* 각 페이지에 상응하는 임베딩 벡터값 필요!

In [25]:
# log_prse['PG_EMB_VCT'] = 0

### 로그데이터 확인

In [26]:
grp_sess = pd.DataFrame(log_prse.groupby('SESS_ID'))
grp_sess.columns = ['SESS_ID', 'LOG']
print(grp_sess.shape)
grp_sess.head(2)

(560510, 2)


,SESS_ID,LOG
0,000117b4880909a482cbfb7a65ddf0f71722293785,STND_YMD WEB_APP_CL_CD \ 2776797 ...
1,000117b4880909a482cbfb7a65ddf0f71722298631,STND_YMD WEB_APP_CL_CD \ 722067 ...


### 로그 전처리

* 날짜 간격 2일이상 -> 다른 세션으로 분리 / sess_Start를 기준으로 세션 분리
* 포인트 관련 데이터 삭제
* 앱여부 0,1

In [27]:
# 데이터 코딩 / 오래걸림 엄청 다시 안돌리게 주의..
len_grp_sess = len(grp_sess)

for i in tqdm(range(len_grp_sess)):
    grp_sess['LOG'][i] = grp_sess['LOG'][i] 
    grp_sess['LOG'][i]['LOG_DTTM'] = pd.to_datetime(grp_sess['LOG'][i]['LOG_DTTM'])
    grp_sess['LOG'][i] = grp_sess['LOG'][i].sort_values(by='LOG_DTTM')
    grp_sess['LOG'][i] = grp_sess['LOG'][i].reset_index(drop=True)
    
    if (grp_sess['LOG'][i]['WEB_APP_CL_CD'] == 'APP').any():
        grp_sess['LOG'][i]['APP'] = 1
    else:
        grp_sess['LOG'][i]['APP'] = 0

 24%|██▎       | 132387/560510 [10:08:54<32:49:09,  3.62it/s]


KeyboardInterrupt: 

In [ ]:
temp = grp_sess['LOG'][0]
temp.iloc[0]['prd_sal_prc']

nan

In [ ]:
type(temp.iloc[0]['prd_sal_prc'])

float

In [ ]:
# 데이터 인디케이팅 -> incs부분 수정
SESS_TIME_list = []
SRCH_EFRT_list = []
PRDV_CNT_list = []
CAT_CNT_list = []
SAME_PAGE_CNT_list = []
EVNT_CNT_list = []
incs_no = []
lst_sess_time = []
night_list = []
SAL_PRD_list = []

for i in tqdm(range(len_grp_sess)):
    temp = grp_sess['LOG'][i]

    # # 전체 세션 시간(time) -> 안쓰는게 나을듯
    # SESS_TIME_list.append(np.sum(temp['PG_STAY_TIME']))

    # 탐색 노력(전체 세션 개수)
    SRCH_EFRT_list.append(len(temp))

    # 상품 뷰 숫자
    PRDV_CNT_list.append((len(temp[temp['PRD_INFO'].notna()])))

    # 카테고리 뷰 숫자
    CAT_CNT_list.append(len(set(temp[temp['PRD_INFO'].notna()]['prd_tp_cat_vl'])) - 1)

    # # 동일한 페이지를 본 숫자
    # SAME_PAGE_CNT_list.append(len(set(temp['PG_URL'])))

    # 이벤트 페이지 탐색 횟수 확인
    EVNT_CNT_list.append(len(temp[temp['EVNT_NM'].notna()]))

    # 탐색 상품가격
    SAL_PRD_list.append(sum(temp[temp['prd_sal_prc'].notna()].prd_sal_prc.apply(int)))

    incs_no.append(temp.iloc[0]['INCS_NO'])    
    lst_sess_time.append(temp.iloc[-1]['LOG_DTTM'])

100%|██████████| 239080/239080 [24:33<00:00, 162.22it/s]


In [ ]:
app_list = []

for i in tqdm(range(len_grp_sess)):
    app_list.append(grp_sess['LOG'][i]['APP'][0])

len(app_list)

  0%|          | 0/239080 [00:00<?, ?it/s]

100%|██████████| 239080/239080 [00:45<00:00, 5306.02it/s] 


239080

In [ ]:
sess_indicate = pd.DataFrame({'SESS_ID' : grp_sess['SESS_ID'],
        # 'SESS_TIME':SESS_TIME_list,
        'SRCH_EFRT' : SRCH_EFRT_list,
        'PRDV_CNT' : PRDV_CNT_list,
        'CAT_CNT' : CAT_CNT_list,
        # 'SAMGE_PAGE_CNT' : SAME_PAGE_CNT_list,
        'EVNT_CNT' : EVNT_CNT_list,
        'SAL_PRD' : SAL_PRD_list,
        'INCS_NO' : incs_no,
        'LST_SESS_TIME' : lst_sess_time,
        'APP' : app_list
        })

print(sess_indicate.shape)
sess_indicate.head(2)

(239080, 9)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP
0,000211c41b2c524ea758d997901733fe1720001995,5,0,-1,5,0,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-03 19:21:01.213,1
1,000211c41b2c524ea758d997901733fe1720067045,3,1,0,3,6300,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-04 13:24:16.188,1


### SCROLL depth 추가

In [ ]:
scroll = pd.read_csv('./raw/SCROLL.csv')
print(scroll.shape)
scroll.head(2)

(6152314, 3)


,SESS_ID,INCS_NO,EVNT_ACTN
0,665c203f95593e4c75756cd84ecd999c1719015429,c3e68bb407daefe878de2e184c8b04cdd903a1fc8a433d...,40%
1,665c203f95593e4c75756cd84ecd999c1719015429,c3e68bb407daefe878de2e184c8b04cdd903a1fc8a433d...,20%


In [ ]:
def del_per(text):
    text = int(text.split('%')[0])
    return text

In [ ]:
# from scipy.stats import nbinom

# mean_x = 39.7
# var_x = 27.1

# p = mean_x / var_x
# r = mean_x * p / (1-p)


# # x = range(2987269)
# x = range(100)
# pmf_vals = nbinom.pmf(x, r, p)

# plt.bar(x, pmf_vals)

In [ ]:
scroll['EVNT_ACTN'] = scroll['EVNT_ACTN'].apply(del_per)
scroll.head(2)

,SESS_ID,INCS_NO,EVNT_ACTN
0,665c203f95593e4c75756cd84ecd999c1719015429,c3e68bb407daefe878de2e184c8b04cdd903a1fc8a433d...,40
1,665c203f95593e4c75756cd84ecd999c1719015429,c3e68bb407daefe878de2e184c8b04cdd903a1fc8a433d...,20


In [ ]:
scroll_grp = scroll.groupby('SESS_ID').agg(AVG_DEPTH=('EVNT_ACTN', 'mean')).reset_index()
print(scroll_grp.shape)
scroll_grp.head(2)

(432110, 2)


,SESS_ID,AVG_DEPTH
0,000117b4880909a482cbfb7a65ddf0f71722298631,26.250000
1,000117b4880909a482cbfb7a65ddf0f71722301833,44.677419


In [ ]:
sess_scr = pd.merge(sess_indicate, scroll_grp, on='SESS_ID', how='left')

# AVG_DEPTH가 없는 값은 0으로 채우기
sess_scr['AVG_DEPTH'] = sess_scr['AVG_DEPTH'].fillna(0)
sess_scr.head(2)

,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP,AVG_DEPTH
0,000211c41b2c524ea758d997901733fe1720001995,5,0,-1,5,0,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-03 19:21:01.213,1,0.0
1,000211c41b2c524ea758d997901733fe1720067045,3,1,0,3,6300,1d061118266aaac746e4418b030d8d10046fd4a4adabe0...,2024-07-04 13:24:16.188,1,0.0


In [ ]:
print(len(sess_scr))
print(len(sess_scr[sess_scr['AVG_DEPTH'] != 0]))

239080
40778


In [ ]:
sess_indicate = sess_scr[sess_scr['AVG_DEPTH'] != 0]

### day_off 추가

In [ ]:
def day_off_check(dt):
    dt = dt.time()
    evening_start = pd.Timestamp('18:00:00').time()
    morning_end = pd.Timestamp('06:00:00').time()
    
    # 저녁 6시 이후 또는 오전 6시 이전인지 확인
    if dt >= evening_start or dt < morning_end:
        return 1
    else:
        return 0

In [ ]:
sess_indicate['day_off'] = sess_indicate['LST_SESS_TIME'].apply(day_off_check)
sess_indicate.head(2)

,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,EVNT_CNT,SAL_PRD,INCS_NO,LST_SESS_TIME,APP,AVG_DEPTH,day_off
39,001ef1290ae23824ae5e97a9fa2a00f31719788953,42,12,1,42,139080,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-01 08:24:21.803,1,13.571429,0
40,001ef1290ae23824ae5e97a9fa2a00f31719926158,5,1,0,5,0,4acc3c2221d10da113ac56989dba4b615457c61195a1a8...,2024-07-02 22:16:40.917,1,13.333333,1


In [ ]:
# 탐색 노력이 4이상인 데이터만 사용 -> 추후 탐색노력에 대한 영향이 없다는 추가 분석 필요
sess_indicate_cutoff = sess_indicate[sess_indicate["SRCH_EFRT"] > 3]

sess_indicate.to_csv('./data/sess_origin.csv')
sess_indicate_cutoff.to_csv('./data/sess_cutoff.csv')